In [14]:
from veripulse.warm_start_inspect import find_best_grape_run, load_grape_run
from pathlib import Path
from veripulse.pulse import PulseConfig, run_grape_si
from veripulse.gates import rx, rhox, hadamard, hadamardZ, pack_subspace_states, extract_subspace_states, Qobj, operator_to_vector, vector_to_operator
import numpy as np
import numpy as np
from qutip import Qobj, fidelity
 
from veripulse.gates import extract_subspace_states, rhox
from veripulse.sdp import choi_optimise_secret_indep, calc_secret_indep
 

from __future__ import annotations

In [9]:
"""
Run run_grape_si with a warm start from find_best_grape_run().
"""
def repack_amps(ws, angles) -> np.ndarray:
    """
    Repack (K, num_tslots, 2) GRAPE amps into (num_tslots, K*2) joint format.

    Sorts amps to match the angle order passed to pack_subspace_states.
    L_ctrl order in nvcenter_system is grouped: [Lx_0,...,Lx_K, Ly_0,...,Ly_K]
    """
    amps   = ws.amps                   # (K, num_tslots, 2)
    thetas = np.array(ws.raw["state_labels"])
    angles = np.array(angles)
    K      = amps.shape[0]

    # match each angle to its JSON index
    sort_idx    = [np.argmin(np.abs(thetas - a)) for a in angles]
    amps_sorted = amps[sort_idx]       # (K, num_tslots, 2) in angles order

    amps_x     = amps_sorted[:, :, 0].T     # (num_tslots, K)
    amps_y     = amps_sorted[:, :, 1].T     # (num_tslots, K)
    amps_joint = np.hstack([amps_x, amps_y])  # (num_tslots, K*2)

    print(f"[repack] sorted labels : {[f'{thetas[i]:.4f}' for i in sort_idx]}")
    print(f"[repack] amps_joint    : {amps_joint.shape}")
    return amps_joint


def run_with_warm_start(ws, angles, init_state, target_state, U, lam,
                        **config_overrides):
    """
    Run run_grape_si using config and amps from a WarmStartResult.

    Parameters
    ----------
    ws               : WarmStartResult from find_best_grape_run()
    angles           : sorted angles passed to pack_subspace_states
    init_state       : vectorised initial state (from pack_subspace_states)
    target_state     : vectorised target state
    U                : block-diagonal unitary
    lam              : secret-independence weight
    **config_overrides : any PulseConfig field to override, e.g.
                         max_iter=2000, max_wall_time=3600, fid_err_targ=1e-12
    """
    # start from ws config, apply overrides
    cfg_dict = dict(ws.config)
    cfg_dict.update(config_overrides)
    cfg = PulseConfig(**cfg_dict)

    if config_overrides:
        print(f"[warm start] config overrides : {config_overrides}")

    amps_joint = repack_amps(ws, angles)

    print(f"[warm start] amps from     : {ws.path.name}")
    print(f"[warm start] si            : {ws.si:.3e}")
    print(f"[warm start] max_iter      : {cfg.max_iter}")
    print(f"[warm start] max_wall_time : {cfg.max_wall_time}")
    print(f"[warm start] fid_err_targ  : {cfg.fid_err_targ}")

    return run_grape_si(
        init_state, target_state,
        U=U, lam=lam,
        config=cfg,
        amps=amps_joint,
    )


 
def compute_si_from_result(result, angles, rho_init):
    """
    Recompute true SI from final evolved states using Choi SDP.
 
    Parameters
    ----------
    result   : qutip-ctrl OptimResult from run_grape_si / run_with_warm_start
    angles   : sorted angles passed to pack_subspace_states
    rho_init : single-qubit initial state (Qobj), e.g. Qobj([[1,0],[0,0]])
 
    Returns
    -------
    res_choi : SecretIndep (has .objective = SI)
    si_lb    : float, lower bound on SI
    """
    K        = len(angles)
    rho_targs    = [Qobj(rhox(a)) for a in angles]
    rho_fins     = extract_subspace_states(result, K)
    rho_fins_np  = [rf.full() if isinstance(rf, Qobj) else rf for rf in rho_fins]
    rho_targs_np = [r.full() for r in rho_targs]
 
    # Choi SDP
    res_choi = choi_optimise_secret_indep(rho_targs_np, rho_fins_np)
    si_lb    = calc_secret_indep(rho_targs_np, rho_fins_np)
 
    # per-state fidelity
    print(f"\n{'─'*50}")
    print(f"{'state':<6} {'angle':>8}  {'err_ul':>10}")
    print(f"{'─'*50}")
    for j, a in enumerate(angles):
        rf  = rho_fins[j] if isinstance(rho_fins[j], Qobj) else Qobj(rho_fins[j])
        err = 1 - fidelity(rho_targs[j], rf)
        print(f"  {j:<4} {a:>8.4f}  {err:>10.3e}")
    print(f"{'─'*50}")
    print(f"  SI (Choi SDP) : {res_choi.objective:.3e}")
    print(f"  SI (lower bd) : {si_lb:.3e}")
    print(f"  fid_err (raw) : {result.fid_err:.3e}")
    print(f"{'─'*50}")
 
    return res_choi, si_lb


In [3]:
# data folder
data_dir_dummyyes = Path.cwd().parent/"data/dummyyes"
data_dir_dummyless = Path.cwd().parent/"data/dummyless"

In [ ]:
# Step 1 — find and inspect
ws = find_best_grape_run(
    data_dir    = data_dir_dummyless,
    num_tslots  = 40,
    detuning    = 0.0,
    drive_error = 0.0,
    rank_by     = "err_ul_mean" , # "si" | "err_ul_mean" | "err_ul_median"
    verbose     = False
)

In [5]:
# Manual — you pick from the table
ws = load_grape_run(Path(data_dir_dummyless,"CRAB_p40_det0.00_err0.00-1.json"))

[load_grape_run] loaded: CRAB_p40_det0.00_err0.00-1.json
  si            : 1.186e-01
  err_ul_mean   : 3.326e-02
  amps shape    : (8, 40, 2)


In [7]:
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi, 5*np.pi/4, 3*np.pi/2, 7*np.pi/4]
# dummyless 
rho_init = Qobj([[1, 0], [0, 0]])
rho_targets = [Qobj(rhox(a)) for a in angles]
unitary_rotations=[rx(t) for t in angles]
vRho_init, vRho_target, U_big = pack_subspace_states(
    rotations=unitary_rotations,
    rho_init=rho_init,
)

res = run_with_warm_start(
    ws, angles,
    init_state   = vRho_init,
    target_state = vRho_target,
    U            = U_big,
    lam          = 0.05,
    max_iter     = 2000,
    max_wall_time = 3600,
    fid_err_targ  = 1e-13,
)

[warm start] config overrides : {'max_iter': 2000, 'max_wall_time': 3600, 'fid_err_targ': 1e-13}
[repack] sorted labels : ['0.0000', '0.7854', '1.5708', '2.3562', '3.1416', '3.9270', '4.7124', '5.4978']
[repack] amps_joint    : (40, 16)
[warm start] amps from     : CRAB_p40_det0.00_err0.00-1.json
[warm start] si            : 1.186e-01
[warm start] max_iter      : 2000
[warm start] max_wall_time : 3600
[warm start] fid_err_targ  : 1e-13
Secret Independence =  0.10329817405868141
fidelity error =  0.014437436251122308
Secret Independence =  0.1031685898606775
fidelity error =  0.01442092406067715
Secret Independence =  0.264368367891804
fidelity error =  0.05056545988820959
Secret Independence =  0.008677364171114612
fidelity error =  0.0035292099246104595
Secret Independence =  0.0035925838929487257
fidelity error =  0.0037646489236755655
Secret Independence =  0.0003682232138786293
fidelity error =  0.003014750192719324
Secret Independence =  9.713938001966348e-05
fidelity error =  0.0

In [13]:
res_choi, si_lb = compute_si_from_result(res, angles, rho_init)


──────────────────────────────────────────────────
state     angle      err_ul
──────────────────────────────────────────────────
  0      0.0000   3.541e-04
  1      0.7854   5.421e-04
  2      1.5708   2.520e-03
  3      2.3562   2.923e-03
  4      3.1416   2.846e-03
  5      3.9270   2.474e-03
  6      4.7124   3.590e-03
  7      5.4978   2.429e-03
──────────────────────────────────────────────────
  SI (Choi SDP) : 1.319e-03
  SI (lower bd) : 4.516e-03
  fid_err (raw) : 4.735e-07
──────────────────────────────────────────────────


/users/home/gustiani/VeriPulse/.venv/lib/python3.14/site-packages/qutip/core/data/expm.py:146: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  return Dense(scipy.linalg.sqrtm(matrix.as_ndarray()))
